# Tokenization Tutorial: From Fundamentals to Tiktoken

## Introduction

**Tokenization** is the process of breaking down text into smaller units called **tokens**. These tokens can be words, subwords, or even individual characters. Tokenization is a fundamental step in Natural Language Processing (NLP) and is crucial for working with Large Language Models (LLMs).

### Why is Tokenization Important?

1. **Text Representation**: Computers don't understand text directly. Tokenization converts text into numerical representations that models can process.
2. **Vocabulary Management**: It helps create a manageable vocabulary size for models.
3. **Cost Estimation**: LLM APIs charge based on the number of tokens processed.
4. **Input Validation**: Models have maximum token limits (e.g., GPT-4 has different context windows).
5. **Efficiency**: Good tokenization balances vocabulary size with representation quality.

In this tutorial, we'll explore:
- Basic tokenization using regular expressions
- Byte Pair Encoding (BPE) algorithm
- Practical tokenization with `tiktoken`
- Real-world applications: cost estimation and input validation

## Part 1: Basic Tokenization with Regular Expressions

Let's start with the simplest form of tokenization: splitting text by spaces.

In [1]:
import re
from collections import Counter

# Sample text
text = """Natural language processing is amazing! 
Processing text helps us understand language. 
Language models need tokenization."""

print("Original text:")
print(text)

Original text:
Natural language processing is amazing! 
Processing text helps us understand language. 
Language models need tokenization.


### Simple Space-Based Tokenization

The most basic approach is to split text by whitespace and remove punctuation.

In [5]:
# Tokenize by splitting on whitespace and converting to lowercase
# We'll also remove punctuation for simplicity
tokens = re.findall(r'\b\w+\b', text.lower())

print(f"\nTokens ({len(tokens)} total):")
print(tokens)


Tokens (15 total):
['natural', 'language', 'processing', 'is', 'amazing', 'processing', 'text', 'helps', 'us', 'understand', 'language', 'language', 'models', 'need', 'tokenization']


### Building a Vocabulary

A **vocabulary** is the set of unique tokens. Let's extract unique words and count their frequencies.

In [6]:
# Get unique tokens (vocabulary)
vocabulary = sorted(set(tokens))

print(f"\nVocabulary ({len(vocabulary)} unique tokens):")
print(vocabulary)

# Count token frequencies
token_counts = Counter(tokens)
print("\nToken frequencies:")
for token, count in token_counts.most_common():
    print(f"  '{token}': {count}")


Vocabulary (12 unique tokens):
['amazing', 'helps', 'is', 'language', 'models', 'natural', 'need', 'processing', 'text', 'tokenization', 'understand', 'us']

Token frequencies:
  'language': 3
  'processing': 2
  'natural': 1
  'is': 1
  'amazing': 1
  'text': 1
  'helps': 1
  'us': 1
  'understand': 1
  'models': 1
  'need': 1
  'tokenization': 1


### Creating Encode and Decode Functions

Now let's create a simple encoder and decoder using Python dictionaries. The encoder maps tokens to integers, and the decoder does the reverse.

In [7]:
# Create token-to-id mapping (encoder)
token_to_id = {token: idx for idx, token in enumerate(vocabulary)}

# Create id-to-token mapping (decoder)
id_to_token = {idx: token for token, idx in token_to_id.items()}

print("Token to ID mapping:")
for token, idx in sorted(token_to_id.items()):
    print(f"  '{token}' -> {idx}")

Token to ID mapping:
  'amazing' -> 0
  'helps' -> 1
  'is' -> 2
  'language' -> 3
  'models' -> 4
  'natural' -> 5
  'need' -> 6
  'processing' -> 7
  'text' -> 8
  'tokenization' -> 9
  'understand' -> 10
  'us' -> 11


In [52]:
token_to_id

{'amazing': 0,
 'helps': 1,
 'is': 2,
 'language': 3,
 'models': 4,
 'natural': 5,
 'need': 6,
 'processing': 7,
 'text': 8,
 'tokenization': 9,
 'understand': 10,
 'us': 11}

In [53]:
id_to_token

{0: 'amazing',
 1: 'helps',
 2: 'is',
 3: 'language',
 4: 'models',
 5: 'natural',
 6: 'need',
 7: 'processing',
 8: 'text',
 9: 'tokenization',
 10: 'understand',
 11: 'us'}

In [12]:
def encode(text, token_to_id):
    """Encode text to a list of token IDs."""
    tokens = re.findall(r'\b\w+\b', text.lower())
    return [token_to_id.get(token, -1) for token in tokens]  # -1 for unknown tokens

def decode(token_ids, id_to_token):
    """Decode a list of token IDs back to text."""
    tokens = [id_to_token.get(idx, '<UNK>') for idx in token_ids]
    return ' '.join(tokens)

# Test encoding and decoding
test_text = "language processing is amazing"
encoded = encode(test_text, token_to_id)
decoded = decode(encoded, id_to_token)

print(f"\nOriginal: {test_text}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")


Original: language processing is amazing
Encoded: [3, 7, 2, 0]
Decoded: language processing is amazing


### Limitations of Simple Tokenization

Let's see what happens with words not in our vocabulary:

In [13]:
# Test with unknown words
new_text = "deep learning and transformers are revolutionary"
encoded_new = encode(new_text, token_to_id)
decoded_new = decode(encoded_new, id_to_token)

print(f"\nNew text: {new_text}")
print(f"Encoded: {encoded_new}")
print(f"Decoded: {decoded_new}")
print("\n⚠️ Notice: Words not in our vocabulary are encoded as -1 and decoded as <UNK>")


New text: deep learning and transformers are revolutionary
Encoded: [-1, -1, -1, -1, -1, -1]
Decoded: <UNK> <UNK> <UNK> <UNK> <UNK> <UNK>

⚠️ Notice: Words not in our vocabulary are encoded as -1 and decoded as <UNK>


**Problems with simple word-based tokenization:**
1. **Unknown words**: Any word not in the vocabulary becomes `<UNK>`
2. **Large vocabulary**: Every unique word needs an entry
3. **No shared meaning**: "run", "running", "runs" are treated as completely different tokens
4. **Rare words**: Uncommon words waste vocabulary space

**Solution**: Subword tokenization algorithms like Byte Pair Encoding (BPE)!

## Part 2: Byte Pair Encoding (BPE)

**Byte Pair Encoding (BPE)** is a subword tokenization algorithm that addresses the limitations of word-based tokenization.

### How BPE Works

BPE starts with individual characters and iteratively merges the most frequent pairs of tokens:

1. **Initialize**: Start with all individual characters as tokens
2. **Count pairs**: Find the most frequent pair of adjacent tokens
3. **Merge**: Replace all occurrences of that pair with a new token
4. **Repeat**: Continue until reaching desired vocabulary size

### BPE Example

Let's walk through a simple example:

In [14]:
# Simple BPE demonstration
corpus = [
    "low",
    "lower",
    "newest",
    "widest"
]

print("Corpus:")
for word in corpus:
    print(f"  {word}")

# Step 1: Start with character-level tokens (with word boundary marker)
# We add a special end-of-word marker </w> to preserve word boundaries
tokenized_corpus = [' '.join(list(word) + ['</w>']) for word in corpus]

print("\nInitial character-level tokenization:")
for word in tokenized_corpus:
    print(f"  {word}")

Corpus:
  low
  lower
  newest
  widest

Initial character-level tokenization:
  l o w </w>
  l o w e r </w>
  n e w e s t </w>
  w i d e s t </w>


### BPE Merge Iterations

Now let's manually demonstrate a few BPE merge operations:

In [15]:
def get_pair_frequencies(tokenized_corpus):
    """Count frequencies of adjacent token pairs."""
    pairs = Counter()
    for word in tokenized_corpus:
        tokens = word.split()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i+1])] += 1
    return pairs

def merge_pair(tokenized_corpus, pair):
    """Merge all occurrences of a token pair."""
    new_corpus = []
    pair_str = ' '.join(pair)
    merged = ''.join(pair)
    
    for word in tokenized_corpus:
        new_word = word.replace(pair_str, merged)
        new_corpus.append(new_word)
    
    return new_corpus

# Iteration 1
print("\n=== BPE Iteration 1 ===")
pairs = get_pair_frequencies(tokenized_corpus)
print("\nMost frequent pairs:")
for pair, count in pairs.most_common(5):
    print(f"  {pair}: {count}")

most_frequent = pairs.most_common(1)[0][0]
print(f"\nMerging: {most_frequent}")
tokenized_corpus = merge_pair(tokenized_corpus, most_frequent)

print("\nAfter merge:")
for word in tokenized_corpus:
    print(f"  {word}")


=== BPE Iteration 1 ===

Most frequent pairs:
  ('l', 'o'): 2
  ('o', 'w'): 2
  ('w', 'e'): 2
  ('e', 's'): 2
  ('s', 't'): 2

Merging: ('l', 'o')

After merge:
  lo w </w>
  lo w e r </w>
  n e w e s t </w>
  w i d e s t </w>


In [16]:
# Iteration 2
print("\n=== BPE Iteration 2 ===")
pairs = get_pair_frequencies(tokenized_corpus)
print("\nMost frequent pairs:")
for pair, count in pairs.most_common(5):
    print(f"  {pair}: {count}")

most_frequent = pairs.most_common(1)[0][0]
print(f"\nMerging: {most_frequent}")
tokenized_corpus = merge_pair(tokenized_corpus, most_frequent)

print("\nAfter merge:")
for word in tokenized_corpus:
    print(f"  {word}")


=== BPE Iteration 2 ===

Most frequent pairs:
  ('lo', 'w'): 2
  ('w', 'e'): 2
  ('e', 's'): 2
  ('s', 't'): 2
  ('t', '</w>'): 2

Merging: ('lo', 'w')

After merge:
  low </w>
  low e r </w>
  n e w e s t </w>
  w i d e s t </w>


In [17]:
# Iteration 3
print("\n=== BPE Iteration 3 ===")
pairs = get_pair_frequencies(tokenized_corpus)
print("\nMost frequent pairs:")
for pair, count in pairs.most_common(5):
    print(f"  {pair}: {count}")

most_frequent = pairs.most_common(1)[0][0]
print(f"\nMerging: {most_frequent}")
tokenized_corpus = merge_pair(tokenized_corpus, most_frequent)

print("\nAfter merge:")
for word in tokenized_corpus:
    print(f"  {word}")


=== BPE Iteration 3 ===

Most frequent pairs:
  ('e', 's'): 2
  ('s', 't'): 2
  ('t', '</w>'): 2
  ('low', '</w>'): 1
  ('low', 'e'): 1

Merging: ('e', 's')

After merge:
  low </w>
  low e r </w>
  n e w es t </w>
  w i d es t </w>


### Advantages of BPE

*  **No unknown words**: Any word can be represented by character combinations  
*  **Efficient vocabulary**: Common subwords are learned, rare words are split  
*  **Shared representations**: "run", "running", "runs" share the "run" subword  
*  **Flexible**: Works across languages and domains  

### BPE in Modern LLMs

Modern language models like GPT use variants of BPE:
- **GPT-2/GPT-3**: Byte-level BPE
- **GPT-4**: Enhanced tokenizer with better multilingual support
- **Typical vocabulary size**: 50,000 - 100,000 tokens

Now let's see how to use these tokenizers in practice!

## Part 3: Practical Tokenization with Tiktoken

`tiktoken` is a fast BPE tokenizer library created by OpenAI for use with their models. It's the same tokenizer used by GPT models.

### Installation

First, let's make sure tiktoken is installed:

In [19]:
# Install tiktoken if not already installed
!uv pip install tiktoken

Using Python 3.10.18 environment at: /Users/tarekatwan/Repos/MyWork/Teach/repos/advanced_machine_learning/.venv
Audited 1 package in 4ms


In [20]:
import tiktoken

print(f"Tiktoken version: {tiktoken.__version__}")

Tiktoken version: 0.12.0


### Available Encodings

Tiktoken supports different encodings for different OpenAI models:

In [24]:
# List available encodings
print("Available encodings:")
for encoding_name in tiktoken.list_encoding_names():
    print(f"  - {encoding_name}")

# Common model-to-encoding mappings
print("\nCommon model encodings:")
models = [
    "gpt-4",
    "gpt-4-turbo",
    "gpt-3.5-turbo",
    "text-davinci-003",
    "text-embedding-ada-002"
]

for model in models:
    try:
        encoding = tiktoken.encoding_for_model(model)
        print(f"  {model}: {encoding.name}")
    except KeyError:
        print(f"  {model}: encoding not found")

Available encodings:
  - gpt2
  - r50k_base
  - p50k_base
  - p50k_edit
  - cl100k_base
  - o200k_base
  - o200k_harmony

Common model encodings:
  gpt-4: cl100k_base
  gpt-4-turbo: cl100k_base
  gpt-3.5-turbo: cl100k_base
  text-davinci-003: p50k_base
  text-embedding-ada-002: cl100k_base


### Basic Encoding and Decoding

Let's start with the `cl100k_base` encoding used by GPT-4 and GPT-3.5-turbo:

In [25]:
# Get the encoding for GPT-4
encoding = tiktoken.get_encoding("cl100k_base")

# Alternatively, get encoding for a specific model
# encoding = tiktoken.encoding_for_model("gpt-4")

print(f"Using encoding: {encoding.name}")
print(f"Vocabulary size: {encoding.n_vocab:,} tokens")

Using encoding: cl100k_base
Vocabulary size: 100,277 tokens


In [32]:
# Example text
text = "Hello, world! This is a tokenization example using tiktoken."

# Encode text to token IDs
tokens = encoding.encode(text)

print(f"Original text: {text}")
print(f"\nToken IDs: {tokens}")
print(f"Number of tokens: {len(tokens)}")

Original text: Hello, world! This is a tokenization example using tiktoken.

Token IDs: [9906, 11, 1917, 0, 1115, 374, 264, 4037, 2065, 3187, 1701, 87272, 5963, 13]
Number of tokens: 14


In [34]:
# Decode token IDs back to text
decoded_text = encoding.decode(tokens)

print(f"Decoded text: {decoded_text}")
print(f"\nMatch original? {text == decoded_text}")

Decoded text: Hello, world! This is a tokenization example using tiktoken.

Match original? True


### Visualizing Tokens

Let's see how text is split into individual tokens:

In [35]:
def visualize_tokens(text, encoding):
    """Show how text is tokenized."""
    tokens = encoding.encode(text)
    
    print(f"Text: {text}")
    print(f"Tokens: {len(tokens)}\n")
    
    print("Token breakdown:")
    for i, token_id in enumerate(tokens):
        token_bytes = encoding.decode_single_token_bytes(token_id)
        token_str = token_bytes.decode('utf-8', errors='replace')
        print(f"  {i+1}. ID={token_id:6d} | '{token_str}'")

# Example 1: Simple sentence
visualize_tokens("The quick brown fox jumps.", encoding)

Text: The quick brown fox jumps.
Tokens: 6

Token breakdown:
  1. ID=   791 | 'The'
  2. ID=  4062 | ' quick'
  3. ID= 14198 | ' brown'
  4. ID= 39935 | ' fox'
  5. ID= 35308 | ' jumps'
  6. ID=    13 | '.'


In [36]:
# Example 2: Technical text
print("\n" + "="*50 + "\n")
visualize_tokens("Machine learning and artificial intelligence", encoding)



Text: Machine learning and artificial intelligence
Tokens: 5

Token breakdown:
  1. ID= 22333 | 'Machine'
  2. ID=  6975 | ' learning'
  3. ID=   323 | ' and'
  4. ID= 21075 | ' artificial'
  5. ID= 11478 | ' intelligence'


In [ ]:
# Example 3: Numbers and special characters
print("\n" + "="*50 + "\n")
visualize_tokens("Price: $1,234.56 (20% off!)", encoding)

In [37]:
# Example 4: Code
print("\n" + "="*50 + "\n")
code = "def hello_world():\n    print('Hello!')"
visualize_tokens(code, encoding)



Text: def hello_world():
    print('Hello!')
Tokens: 10

Token breakdown:
  1. ID=   755 | 'def'
  2. ID= 24748 | ' hello'
  3. ID= 32892 | '_world'
  4. ID=  4019 | '():
'
  5. ID=   262 | '   '
  6. ID=  1194 | ' print'
  7. ID=   493 | '(''
  8. ID=  9906 | 'Hello'
  9. ID=     0 | '!'
  10. ID=   873 | '')'


### Comparing Different Encodings

Different encodings can produce different token counts for the same text:

In [38]:
text = "Tokenization is fundamental for natural language processing and large language models."

print(f"Text: {text}\n")
print("Token counts by encoding:")

for encoding_name in ["cl100k_base", "p50k_base", "r50k_base"]:
    enc = tiktoken.get_encoding(encoding_name)
    tokens = enc.encode(text)
    print(f"  {encoding_name:15s}: {len(tokens):3d} tokens")

Text: Tokenization is fundamental for natural language processing and large language models.

Token counts by encoding:
  cl100k_base    :  13 tokens
  p50k_base      :  13 tokens
  r50k_base      :  13 tokens


## Part 4: Cost Estimation and Management

One of the most practical applications of tokenization is **estimating API costs**. OpenAI and other LLM providers charge based on the number of tokens processed.

### Understanding Token-Based Pricing

Here is the typical OpenAI pricing (prices may vary):
Check out https://platform.openai.com/docs/pricing for more details.

| Model | Input (per 1M tokens) | Output (per 1M tokens) |
|-------|----------------------|------------------------|
| GPT-4 Turbo | $10.00 | $30.00 |
| GPT-3.5 Turbo | $0.50 | $1.50 |

**Important**: Both input (prompt) and output (completion) tokens are counted!

In [39]:
# Pricing constants (per 1M tokens, in USD)
PRICING = {
    "gpt-4-turbo": {"input": 10.00, "output": 30.00},
    "gpt-3.5-turbo": {"input": 0.50, "output": 1.50},
}

def estimate_cost(text, model="gpt-4-turbo", output_tokens=0):
    """
    Estimate the cost of processing text with a given model.
    
    Args:
        text: Input text (prompt)
        model: Model name
        output_tokens: Expected number of output tokens
    
    Returns:
        Dictionary with token counts and cost breakdown
    """
    encoding = tiktoken.encoding_for_model(model)
    input_tokens = len(encoding.encode(text))
    
    # Calculate costs
    input_cost = (input_tokens / 1_000_000) * PRICING[model]["input"]
    output_cost = (output_tokens / 1_000_000) * PRICING[model]["output"]
    total_cost = input_cost + output_cost
    
    return {
        "model": model,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }

def print_cost_estimate(estimate):
    """Pretty print cost estimate."""
    print(f"Model: {estimate['model']}")
    print(f"\nTokens:")
    print(f"  Input:  {estimate['input_tokens']:,}")
    print(f"  Output: {estimate['output_tokens']:,}")
    print(f"  Total:  {estimate['total_tokens']:,}")
    print(f"\nCost:")
    print(f"  Input:  ${estimate['input_cost']:.6f}")
    print(f"  Output: ${estimate['output_cost']:.6f}")
    print(f"  Total:  ${estimate['total_cost']:.6f}")

### Example 1: Simple Query

In [40]:
prompt = "What is machine learning? Explain in 2-3 sentences."

# Estimate assuming ~50 tokens in response
estimate = estimate_cost(prompt, model="gpt-4-turbo", output_tokens=50)
print_cost_estimate(estimate)

Model: gpt-4-turbo

Tokens:
  Input:  13
  Output: 50
  Total:  63

Cost:
  Input:  $0.000130
  Output: $0.001500
  Total:  $0.001630


### Example 2: Document Summarization

In [41]:
# Simulating a longer document
document = """Artificial intelligence (AI) is transforming industries worldwide. 
Machine learning, a subset of AI, enables computers to learn from data without 
explicit programming. Deep learning, using neural networks with multiple layers, 
has achieved remarkable results in image recognition, natural language processing, 
and game playing. Large language models like GPT-4 can generate human-like text, 
answer questions, write code, and assist with various tasks. However, these models 
also raise important questions about ethics, bias, and responsible AI development. 
As AI continues to advance, it's crucial to ensure these technologies benefit 
humanity while minimizing potential risks."""

prompt = f"Summarize the following text in 3 bullet points:\n\n{document}"

# Estimate assuming ~100 tokens in response
estimate = estimate_cost(prompt, model="gpt-4-turbo", output_tokens=100)
print_cost_estimate(estimate)

Model: gpt-4-turbo

Tokens:
  Input:  140
  Output: 100
  Total:  240

Cost:
  Input:  $0.001400
  Output: $0.003000
  Total:  $0.004400


### Example 3: Comparing Model Costs

In [44]:
prompt = "Write a Python function to calculate the Fibonacci sequence up to n terms."
expected_output = 150  # tokens

print("Cost comparison for the same task:\n")

for model in ["gpt-4-turbo", "gpt-3.5-turbo"]:
    estimate = estimate_cost(prompt, model=model, output_tokens=expected_output)
    print(f"{model}:")
    print(f"  Total tokens: {estimate['total_tokens']:,}")
    print(f"  Total cost: ${estimate['total_cost']:.6f}")
    print()

Cost comparison for the same task:

gpt-4-turbo:
  Total tokens: 164
  Total cost: $0.004640

gpt-3.5-turbo:
  Total tokens: 164
  Total cost: $0.000232



### Example 4: Batch Processing Cost Estimation

In [45]:
# Simulating processing multiple customer reviews
reviews = [
    "Great product! Highly recommend.",
    "Terrible experience. Would not buy again.",
    "Average quality for the price.",
    "Exceeded my expectations. Five stars!",
    "Not worth the money. Very disappointed."
]

system_prompt = "Classify the sentiment of the following review as positive, negative, or neutral."
expected_output_per_review = 5  # Just a few tokens for classification

total_input_tokens = 0
total_output_tokens = 0

encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
system_tokens = len(encoding.encode(system_prompt))

for review in reviews:
    review_tokens = len(encoding.encode(review))
    total_input_tokens += system_tokens + review_tokens
    total_output_tokens += expected_output_per_review

input_cost = (total_input_tokens / 1_000_000) * PRICING["gpt-3.5-turbo"]["input"]
output_cost = (total_output_tokens / 1_000_000) * PRICING["gpt-3.5-turbo"]["output"]
total_cost = input_cost + output_cost

print(f"Batch processing {len(reviews)} reviews:")
print(f"\nTotal tokens:")
print(f"  Input:  {total_input_tokens:,}")
print(f"  Output: {total_output_tokens:,}")
print(f"  Total:  {total_input_tokens + total_output_tokens:,}")
print(f"\nTotal cost: ${total_cost:.6f}")
print(f"Cost per review: ${total_cost/len(reviews):.6f}")

# Scale up
num_reviews = 10000
scaled_cost = total_cost * (num_reviews / len(reviews))
print(f"\nEstimated cost for {num_reviews:,} reviews: ${scaled_cost:.2f}")

Batch processing 5 reviews:

Total tokens:
  Input:  117
  Output: 25
  Total:  142

Total cost: $0.000096
Cost per review: $0.000019

Estimated cost for 10,000 reviews: $0.19


### Cost Optimization Tips

💡 **Strategies to reduce token usage and costs:**

1. **Use concise prompts**: Remove unnecessary words
2. **Choose the right model**: Use GPT-3.5 for simpler tasks
3. **Limit output length**: Use `max_tokens` parameter
4. **Cache results**: Don't reprocess the same input
5. **Batch efficiently**: Group similar requests
6. **Preprocess text**: Remove redundant information

## Part 5: Input Length Validation

Every LLM has a **maximum context window** - the maximum number of tokens it can process at once. Exceeding this limit will cause errors.

### Common Context Windows

| Model | Max Tokens |
|-------|------------|
| GPT-4 Turbo | 128,000 |
| GPT-4 | 8,192 |
| GPT-3.5 Turbo | 16,385 |
| GPT-3.5 Turbo (older) | 4,096 |

**Important**: The context window includes both input (prompt) AND output (completion) tokens!

In [46]:
# Model context limits
CONTEXT_LIMITS = {
    "gpt-4-turbo": 128000,
    "gpt-4": 8192,
    "gpt-3.5-turbo": 16385,
}

def validate_input_length(text, model="gpt-4-turbo", max_output_tokens=1000):
    """
    Validate if input text fits within model's context window.
    
    Args:
        text: Input text to validate
        model: Model name
        max_output_tokens: Maximum expected output tokens
    
    Returns:
        Dictionary with validation results
    """
    encoding = tiktoken.encoding_for_model(model)
    input_tokens = len(encoding.encode(text))
    total_tokens = input_tokens + max_output_tokens
    context_limit = CONTEXT_LIMITS[model]
    
    is_valid = total_tokens <= context_limit
    tokens_over = max(0, total_tokens - context_limit)
    utilization = (total_tokens / context_limit) * 100
    
    return {
        "model": model,
        "input_tokens": input_tokens,
        "max_output_tokens": max_output_tokens,
        "total_tokens": total_tokens,
        "context_limit": context_limit,
        "is_valid": is_valid,
        "tokens_over": tokens_over,
        "utilization_percent": utilization
    }

def print_validation(result):
    """Pretty print validation results."""
    print(f"Model: {result['model']}")
    print(f"Context limit: {result['context_limit']:,} tokens")
    print(f"\nToken usage:")
    print(f"  Input:  {result['input_tokens']:,}")
    print(f"  Output: {result['max_output_tokens']:,} (max)")
    print(f"  Total:  {result['total_tokens']:,}")
    print(f"\nUtilization: {result['utilization_percent']:.1f}%")
    
    if result['is_valid']:
        print(f"\n✅ Valid: Input fits within context window")
        remaining = result['context_limit'] - result['total_tokens']
        print(f"   Remaining capacity: {remaining:,} tokens")
    else:
        print(f"\n❌ Invalid: Exceeds context window by {result['tokens_over']:,} tokens")
        print(f"   Need to reduce input or use a model with larger context")

### Example 1: Short Input (Valid)

In [47]:
short_text = "Explain quantum computing in simple terms."

result = validate_input_length(short_text, model="gpt-4", max_output_tokens=500)
print_validation(result)

Model: gpt-4
Context limit: 8,192 tokens

Token usage:
  Input:  8
  Output: 500 (max)
  Total:  508

Utilization: 6.2%

✅ Valid: Input fits within context window
   Remaining capacity: 7,684 tokens


### Example 2: Long Document

In [48]:
# Simulate a long document by repeating text
base_text = """Artificial intelligence and machine learning are revolutionizing 
various industries. Deep learning models have achieved remarkable success in 
computer vision, natural language processing, and reinforcement learning. """

long_document = base_text * 100  # Repeat to make it longer

print(f"Document length: {len(long_document):,} characters\n")

# Test with GPT-3.5 Turbo (older version with 4K limit)
# For demonstration, let's manually set a 4K limit
CONTEXT_LIMITS["gpt-3.5-turbo-4k"] = 4096

result = validate_input_length(long_document, model="gpt-3.5-turbo", max_output_tokens=1000)
print_validation(result)

Document length: 21,800 characters

Model: gpt-3.5-turbo
Context limit: 16,385 tokens

Token usage:
  Input:  3,302
  Output: 1,000 (max)
  Total:  4,302

Utilization: 26.3%

✅ Valid: Input fits within context window
   Remaining capacity: 12,083 tokens


### Example 3: Comparing Models for Same Input

In [49]:
medium_text = base_text * 50

print("Validation across different models:\n")
print("=" * 60)

for model in ["gpt-4", "gpt-3.5-turbo", "gpt-4-turbo"]:
    result = validate_input_length(medium_text, model=model, max_output_tokens=1000)
    print(f"\n{model}:")
    print(f"  Limit: {result['context_limit']:,} tokens")
    print(f"  Usage: {result['total_tokens']:,} tokens ({result['utilization_percent']:.1f}%)")
    print(f"  Status: {'✅ Valid' if result['is_valid'] else '❌ Invalid'}")

Validation across different models:


gpt-4:
  Limit: 8,192 tokens
  Usage: 2,652 tokens (32.4%)
  Status: ✅ Valid

gpt-3.5-turbo:
  Limit: 16,385 tokens
  Usage: 2,652 tokens (16.2%)
  Status: ✅ Valid

gpt-4-turbo:
  Limit: 128,000 tokens
  Usage: 2,652 tokens (2.1%)
  Status: ✅ Valid


### Example 4: Dynamic Text Truncation

In [50]:
def truncate_to_limit(text, model="gpt-4", max_output_tokens=1000, safety_margin=100):
    """
    Truncate text to fit within model's context window.
    
    Args:
        text: Input text
        model: Model name
        max_output_tokens: Maximum expected output tokens
        safety_margin: Extra tokens to reserve
    
    Returns:
        Truncated text that fits within limits
    """
    encoding = tiktoken.encoding_for_model(model)
    context_limit = CONTEXT_LIMITS[model]
    
    # Calculate maximum allowed input tokens
    max_input_tokens = context_limit - max_output_tokens - safety_margin
    
    # Encode text
    tokens = encoding.encode(text)
    
    # Truncate if necessary
    if len(tokens) > max_input_tokens:
        tokens = tokens[:max_input_tokens]
        truncated_text = encoding.decode(tokens)
        was_truncated = True
    else:
        truncated_text = text
        was_truncated = False
    
    return {
        "text": truncated_text,
        "was_truncated": was_truncated,
        "original_tokens": len(encoding.encode(text)),
        "final_tokens": len(tokens),
        "tokens_removed": len(encoding.encode(text)) - len(tokens)
    }

# Test truncation
very_long_text = base_text * 200

result = truncate_to_limit(very_long_text, model="gpt-4", max_output_tokens=1000)

print("Truncation results:")
print(f"  Original tokens: {result['original_tokens']:,}")
print(f"  Final tokens: {result['final_tokens']:,}")
print(f"  Tokens removed: {result['tokens_removed']:,}")
print(f"  Was truncated: {result['was_truncated']}")
print(f"\nTruncated text preview:")
print(result['text'][:200] + "...")

Truncation results:
  Original tokens: 6,602
  Final tokens: 6,602
  Tokens removed: 0
  Was truncated: False

Truncated text preview:
Artificial intelligence and machine learning are revolutionizing 
various industries. Deep learning models have achieved remarkable success in 
computer vision, natural language processing, and reinfo...


### Input Validation Best Practices

✅ **Always validate before sending to API**  
✅ **Account for both input and output tokens**  
✅ **Add a safety margin** (100-200 tokens)  
✅ **Choose appropriate model** based on input size  
✅ **Implement truncation strategy** for long inputs  
✅ **Consider chunking** for very long documents  

## Part 6: Practical Exercises

Now it's your turn! Try these exercises to practice what you've learned.

### Exercise 1: Token Counting

Count the tokens in your own text and compare different encodings.

In [58]:
# TODO: Replace with your own text
my_text = "Your text here..."

# Count tokens using different encodings
for encoding_name in ["cl100k_base", "p50k_base"]:
    enc = tiktoken.get_encoding(encoding_name)
    tokens = enc.encode(my_text)
    print(f"{encoding_name}: {len(tokens)} tokens")

cl100k_base: 6 tokens
p50k_base: 6 tokens


### Exercise 2: Cost Calculation

Calculate the cost of processing a batch of customer support tickets.

In [ ]:
# Sample customer support tickets
tickets = [
    "My order hasn't arrived yet. Order #12345.",
    "I need to return a defective product.",
    "Can you help me reset my password?",
    # TODO: Add more tickets
]

# TODO: Calculate total cost for processing all tickets with GPT-3.5-turbo
# Assume each response is ~100 tokens

# Your code here...

### Exercise 3: Input Validation

Write a function that checks if a conversation history fits within the context window.

In [ ]:
# Sample conversation
conversation = [
    {"role": "user", "content": "What is Python?"},
    {"role": "assistant", "content": "Python is a high-level programming language..."},
    {"role": "user", "content": "How do I install it?"},
    # TODO: Add more messages
]

# TODO: Calculate total tokens in conversation
# TODO: Check if it fits in GPT-4's 8K context window

# Your code here...

### Exercise 4: Optimize Prompt Length

Take a verbose prompt and reduce its token count while maintaining meaning.

In [ ]:
verbose_prompt = """I would like you to please help me understand the concept of 
machine learning in a way that is easy to understand. Could you please explain 
it to me using simple language and maybe provide some examples that would help 
me grasp the concept better? I would really appreciate it if you could break 
it down into simple terms."""

# TODO: Rewrite the prompt to use fewer tokens
optimized_prompt = ""  # Your optimized version

# Compare token counts
encoding = tiktoken.get_encoding("cl100k_base")
print(f"Original: {len(encoding.encode(verbose_prompt))} tokens")
print(f"Optimized: {len(encoding.encode(optimized_prompt))} tokens")
print(f"Reduction: {len(encoding.encode(verbose_prompt)) - len(encoding.encode(optimized_prompt))} tokens")

## Summary

In this tutorial, you learned:

### 1. Tokenization Fundamentals
- What tokenization is and why it matters
- Basic regex-based tokenization
- Building vocabularies and encode/decode functions

### 2. Byte Pair Encoding (BPE)
- How BPE works through iterative merging
- Advantages over word-based tokenization
- Why modern LLMs use BPE variants

### 3. Tiktoken Library
- Different encodings for different models
- Encoding and decoding text
- Visualizing token boundaries

### 4. Cost Estimation
- Token-based pricing models
- Calculating API costs
- Comparing costs across models
- Optimization strategies

### 5. Input Validation
- Understanding context windows
- Validating input length
- Truncating text to fit limits
- Best practices

### Key Takeaways

💡 **Always count tokens before calling LLM APIs**  
💡 **Different models use different tokenizers**  
💡 **Both input and output tokens count toward limits and costs**  
💡 **Optimize prompts to reduce costs**  
💡 **Validate inputs to avoid context window errors**  

### Additional Resources

- [Tiktoken GitHub](https://github.com/openai/tiktoken)
- [OpenAI Tokenizer Tool](https://platform.openai.com/tokenizer)
- [OpenAI Pricing](https://openai.com/pricing)
- [BPE Paper](https://arxiv.org/abs/1508.07909)

## Bonus: Interactive Token Explorer

Use this cell to experiment with different texts and see how they're tokenized!

In [51]:
def explore_tokenization(text, model="gpt-4"):
    """Interactive tokenization explorer."""
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(text)
    
    print(f"Model: {model}")
    print(f"Encoding: {encoding.name}")
    print(f"\nText: {text}")
    print(f"\nToken count: {len(tokens)}")
    print(f"Character count: {len(text)}")
    print(f"Tokens per character: {len(tokens)/len(text):.2f}")
    
    print("\nToken breakdown:")
    for i, token_id in enumerate(tokens):
        token_bytes = encoding.decode_single_token_bytes(token_id)
        token_str = token_bytes.decode('utf-8', errors='replace')
        print(f"  {i+1:3d}. [{token_id:6d}] '{token_str}'")
    
    # Cost estimate
    if model in PRICING:
        cost = estimate_cost(text, model=model, output_tokens=0)
        print(f"\nCost (input only): ${cost['input_cost']:.8f}")

# Try it out!
explore_tokenization("Hello, world! 🌍", model="gpt-4")

Model: gpt-4
Encoding: cl100k_base

Text: Hello, world! 🌍

Token count: 7
Character count: 15
Tokens per character: 0.47

Token breakdown:
    1. [  9906] 'Hello'
    2. [    11] ','
    3. [  1917] ' world'
    4. [     0] '!'
    5. [ 11410] ' �'
    6. [   234] '�'
    7. [   235] '�'
